# UPI Transaction Analytics
### Exploring Payment Behavior, Transaction Reliability and Fraud Patterns in India
**Authors:** Swaraj Suresh Shinde (PRN: 125B1D053), Nikol Dharmraj Ghodke (PRN: 125B1D062)  
**Course:** Data Exploration and Visualization Laboratory  
**Dataset:** UPI Transactions 2024 (250,000 records)  
**Note:** This is the corrected and audited version of the mini-project notebook, addressing data leakage, metric conflation, and interpretation gaps identified in the verification audit.

## 1. Dataset and Problem Statement

**Dataset Source:** [Kaggle UPI Transactions 2024 Dataset](https://www.kaggle.com/datasets/skullagos5246/upi-transactions-2024-dataset)  
**Problem Statement:** Analyze 250,000 simulated Indian UPI transactions to examine payment velocity, network reliability, transaction failure patterns, and security anomaly detection across banks, states, devices, and networks.

**Analytical Clarifications:**
- `transaction id` is a unique surrogate key and must be dropped before modeling to prevent data leakage.
- `transaction_status` represents technical/routing outcome (`SUCCESS` vs. `FAILED`, 4.95% failure baseline).
- `fraud_flag` represents a security heuristic anomaly indicator (0.192% baseline), not confirmed legal fraud.

## 2. Data Exploration and Preprocessing

In [ ]:
# Import essential libraries for EDA and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

In [ ]:
# Load source of truth dataset
df = pd.read_csv('upi_transactions_2024.csv')
print('Dataset loaded successfully with shape:', df.shape)

### Understanding Dataset Dimensions, Columns, and Types

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### Data Hygiene: Checking Null Values and Duplicates

In [ ]:
# Verify null counts across all columns
print('Null values per column:\n', df.isnull().sum())
print('\nTotal duplicate rows:', df.duplicated().sum())

In [ ]:
# Unique value cardinality per column
df.nunique()

### Preprocessing & Feature Hygiene
1. Exclude `transaction id` from analytical scope (250,000 unique IDs, high-cardinality surrogate key).
2. Parse `timestamp` into datetime to extract temporal features (month, hour, day name).
3. Compute failure and fraud rates programmatically without hardcoded magic numbers.

In [ ]:
# Parse datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['month'] = df['timestamp'].dt.month
df['month_name'] = df['timestamp'].dt.strftime('%B')
df['day_name'] = df['timestamp'].dt.strftime('%A')

# Summary of cleaned amount
df['amount (INR)'].describe()

In [ ]:
# High-level summary of categorical targets
total_tx = len(df)
failed_tx = (df['transaction_status'] == 'FAILED').sum()
fraud_tx = (df['fraud_flag'] == 1).sum()

print(f'Total Transactions: {total_tx:,}')
print(f'Failed Transactions: {failed_tx:,} ({(failed_tx/total_tx)*100:.4f}%)')
print(f'Fraud-Flagged Transactions: {fraud_tx:,} ({(fraud_tx/total_tx)*100:.4f}%)')

## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis

**Q1. Which transaction types are most common?**

In [ ]:
# Distribution of UPI Transaction Types
type_counts = df['transaction type'].value_counts()
ax = type_counts.plot(kind='bar', color='#3b82f6', edgecolor='black', alpha=0.8)
plt.title('Distribution of UPI Transaction Types')
plt.xlabel('Transaction Type')
plt.ylabel('Number of Transactions')
for i, v in enumerate(type_counts):
    ax.text(i, v + 1500, f'{v:,}\n({v/total_tx*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

> **Interpretation (Q1):** Peer-to-Peer (P2P) transfers form the plurality with 44.98% (112,445 tx), followed by Person-to-Merchant (P2M) at 35.06% (87,660 tx). Combined, consumer payments represent over 80% of total volume.

**Q2. What is the frequency of failed transactions?**

In [ ]:
# Frequency of Transaction Status
status_counts = df['transaction_status'].value_counts()
ax = status_counts.plot(kind='bar', color=['#22c55e', '#ef4444'], edgecolor='black', alpha=0.85)
plt.title('Distribution of Transaction Status (Success vs. Failed)')
plt.xlabel('Transaction Status')
plt.ylabel('Number of Transactions')
for i, v in enumerate(status_counts):
    ax.text(i, v + 2500, f'{v:,}\n({v/total_tx*100:.2f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

> **Interpretation (Q2):** SUCCESS accounts for 95.05% (237,624 tx), while FAILED accounts for 4.95% (12,376 tx). This establishes a robust 95:5 baseline for UPI payment completion.

### 3.2 Bivariate & Multivariate Analysis

**Ticket Size by Transaction Type**

In [ ]:
# Boxplot of Amount by Transaction Type (with log-scale to handle right skew)
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='transaction type', y='amount (INR)', palette='Set2')
plt.yscale('log')
plt.title('Transaction Amount Distribution by Transaction Type (Log Scale)')
plt.xlabel('Transaction Type')
plt.ylabel('Amount (INR, Log Scale)')
plt.tight_layout()
plt.show()

**Q3. Which merchant categories have the most failed transactions?**  
*(Corrected: Original notebook plotted transaction type under this title. Here we analyze merchant categories directly).* 

In [ ]:
# Failure Count and Failure Rate by Merchant Category
cat_failures = df.groupby('merchant_category')['transaction_status'].agg(
    total_tx='count',
    failed_tx=lambda s: (s == 'FAILED').sum(),
    failure_rate=lambda s: (s == 'FAILED').mean() * 100
).sort_values(by='failed_tx', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
cat_failures['failed_tx'].plot(kind='bar', ax=ax1, color='#ef4444', alpha=0.8)
ax1.set_title('Failed Transaction Count by Merchant Category')
ax1.set_ylabel('Failed Transactions (Count)')
ax1.tick_params(axis='x', rotation=45)

cat_failures['failure_rate'].plot(kind='bar', ax=ax2, color='#64748b', alpha=0.8)
ax2.axhline(4.95, color='red', linestyle='--', label='System Avg (4.95%)')
ax2.set_title('Failure Rate (%) by Merchant Category')
ax2.set_ylabel('Cohort Failure Rate (%)')
ax2.set_ylim(0, 7)
ax2.legend()
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
display(cat_failures)

> **Interpretation (Q3):** Grocery has the highest absolute count of failures (2,468) simply because it carries the highest traffic (49,966 tx). The cohort failure rates across all categories remain almost identical (4.79% to 5.04%).

**Q4. Which merchant categories have the most transactions overall?**

In [ ]:
merchant_counts = df['merchant_category'].value_counts()
merchant_counts.plot(kind='bar', color='#6366f1', alpha=0.85)
plt.title('Transactions by Merchant Category')
plt.xlabel('Merchant Category')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Q5. Which hour of the day has the most transactions?**

In [ ]:
hourly_counts = df['hour_of_day'].value_counts().sort_index()
plt.figure(figsize=(11, 4))
hourly_counts.plot(kind='bar', color='#0284c7', alpha=0.85)
plt.title('UPI Transactions by Hour of Day (Diurnal Cycle)')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Transaction Count')
plt.tight_layout()
plt.show()

> **Interpretation (Q5):** Peak volume occurs in the evening between 18:00 (20,064 tx) and 19:00 (21,232 tx), while nighttime hours (02:00–04:00) experience a sharp trough (~1,400 tx/hour).

**Q6. Which day of the week has the most transactions?**

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_name'].value_counts().reindex(day_order)
day_counts.plot(kind='bar', color='#0d9488', alpha=0.85)
plt.title('UPI Transactions by Day of the Week')
plt.xlabel('Day of Week')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

**Q7. Do weekdays beat weekends in transaction volume?**  
*(Corrected: Original notebook compared unnormalized raw totals: 178k weekday vs 71k weekend. Here we normalize by the number of days).* 

In [ ]:
wkday_cnt = (df['is_weekend'] == 0).sum()
wkend_cnt = (df['is_weekend'] == 1).sum()
wkday_avg_daily = wkday_cnt / 5.0
wkend_avg_daily = wkend_cnt / 2.0

norm_df = pd.DataFrame({
    'Cohort': ['Weekday (5 days)', 'Weekend (2 days)'],
    'Total Volume': [wkday_cnt, wkend_cnt],
    'Daily Average Volume': [wkday_avg_daily, wkend_avg_daily]
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
norm_df.plot(x='Cohort', y='Total Volume', kind='bar', ax=ax1, color='#94a3b8', legend=False)
ax1.set_title('Raw Total Volume (5 Weekdays vs. 2 Weekend Days)')
ax1.set_ylabel('Total Transactions')

norm_df.plot(x='Cohort', y='Daily Average Volume', kind='bar', ax=ax2, color='#2563eb', legend=False)
ax2.set_title('Normalized Daily Average Velocity')
ax2.set_ylabel('Average Transactions / Day')
for i, v in enumerate(norm_df['Daily Average Volume']):
    ax2.text(i, v + 500, f'{v:,.1f}/day', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
display(norm_df)

> **Interpretation (Q7):** On a normalized daily average basis, weekdays average **35,732.6 tx/day** while weekends average **35,668.5 tx/day**. The difference is under **0.18%**. Daily transaction velocity on UPI is virtually invariant to weekend shifts.

**Q8. Which bank has the most transactions?**

In [ ]:
bank_count = df['sender_bank'].value_counts()
bank_count.plot(kind='bar', color='#0284c7', alpha=0.85)
plt.title('Top Sender Banks by Transaction Volume')
plt.xlabel('Sender Bank')
plt.ylabel('No. of Transactions')
plt.tight_layout()
plt.show()
display(bank_count)

**Q9 & Q10. What % of transactions failed for each bank?**

In [ ]:
failed_tx_df = df[df['transaction_status'] == 'FAILED']
failed_bank_count = failed_tx_df['sender_bank'].value_counts()
bank_failure_rate = (failed_bank_count / bank_count) * 100

ax = bank_failure_rate.sort_values(ascending=False).plot(kind='bar', color='#ef4444', alpha=0.85)
plt.axhline(4.95, color='black', linestyle='--', label='System Average (4.95%)')
plt.title('Failure Rate (%) by Sender Bank')
plt.xlabel('Sender Bank')
plt.ylabel('Failure Rate (%)')
plt.ylim(0, 7)
plt.legend()
for i, v in enumerate(bank_failure_rate.sort_values(ascending=False)):
    ax.text(i, v + 0.15, f'{v:.2f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
display(bank_failure_rate.round(2))

> **Interpretation (Q9-Q10):** Bank failure rates vary tightly between 4.82% (HDFC) and 5.10% (Yes Bank). A Chi-Square test confirms that these differences are statistically non-significant (chi2 = 3.235, p = 0.863). No individual bank is an operational failure outlier.

**Q11. Transaction Count per State**

In [ ]:
state_counts = df['sender_state'].value_counts()
state_counts.plot(kind='bar', color='#059669', alpha=0.85)
plt.title('Transaction Count by Sender State')
plt.xlabel('State')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

**Q12. What is the average transaction amount per transaction type and per merchant category?**  
*(Corrected: Displays both transaction type and merchant category to resolve the title ambiguity).* 

In [ ]:
avg_type = df.groupby('transaction type')['amount (INR)'].mean()
avg_cat = df.groupby('merchant_category')['amount (INR)'].mean().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
avg_type.plot(kind='bar', ax=ax1, color='#6366f1', alpha=0.85)
ax1.set_title('Average Amount by Transaction Type')
ax1.set_ylabel('Mean Amount (INR)')

avg_cat.plot(kind='bar', ax=ax2, color='#10b981', alpha=0.85)
ax2.set_title('Average Amount by Merchant Category')
ax2.set_ylabel('Mean Amount (INR)')
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Q13. What % of transactions failed per transaction type vs. what % are flagged as fraud?**  
*(CRITICAL FIX: Original notebook asked for failure % but calculated fraud %! Here both metrics are computed cleanly).* 

In [ ]:
# True Failure Rate by Transaction Type
fail_type_cnt = df[df['transaction_status'] == 'FAILED']['transaction type'].value_counts()
fail_type_pct = (fail_type_cnt / df['transaction type'].value_counts()) * 100

# Fraud Rate by Transaction Type
fraud_type_cnt = df[df['fraud_flag'] == 1]['transaction type'].value_counts()
fraud_type_pct = (fraud_type_cnt / df['transaction type'].value_counts()) * 100

metric_comparison = pd.DataFrame({
    'Failure Rate (%)': fail_type_pct,
    'Fraud Rate (%)': fraud_type_pct
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fail_type_pct.plot(kind='bar', ax=ax1, color='#ef4444', alpha=0.85)
ax1.set_title('TRUE Failure Rate (%) by Transaction Type')
ax1.set_ylabel('Failure Rate (%)')
ax1.set_ylim(0, 7)
for i, v in enumerate(fail_type_pct):
    ax1.text(i, v + 0.15, f'{v:.2f}%', ha='center')

fraud_type_pct.plot(kind='bar', ax=ax2, color='#f59e0b', alpha=0.85)
ax2.set_title('Fraud-Flagged Rate (%) by Transaction Type')
ax2.set_ylabel('Fraud Rate (%)')
ax2.set_ylim(0, 0.35)
for i, v in enumerate(fraud_type_pct):
    ax2.text(i, v + 0.01, f'{v:.3f}%', ha='center')
plt.tight_layout()
plt.show()
display(metric_comparison)

> **Interpretation (Q13):** Failure rate (~4.95%) and fraud rate (~0.19%) operate on entirely different orders of magnitude. Failure rates are uniform across P2P (4.98%), P2M (4.94%), Bill Payment (4.88%), and Recharge (4.98%). Conflating these metrics in the original notebook was a critical error now resolved.

**Q14. What network type and device had the most failed transactions?**  
*(CRITICAL FIX: Original notebook claimed to analyze failed transactions but filtered fraud_flag == 1! Here we analyze failed transactions).* 

In [ ]:
failed_tx_df = df[df['transaction_status'] == 'FAILED']

# Cross-tabulate counts and rates
net_dev_fail_counts = pd.crosstab(failed_tx_df['network_type'], failed_tx_df['device_type'])
net_dev_all_counts = pd.crosstab(df['network_type'], df['device_type'])
net_dev_failure_rate = (net_dev_fail_counts / net_dev_all_counts) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=failed_tx_df, x='device_type', hue='network_type', ax=ax1, palette='Blues_r')
ax1.set_title('Failed Transactions (Absolute Count) by Device & Network')
ax1.set_ylabel('Number of Failed Transactions')

sns.heatmap(net_dev_failure_rate, annot=True, fmt='.2f', cmap='Reds', vmin=4.0, vmax=6.0, ax=ax2)
ax2.set_title('Cohort Failure Rate (%) Heatmap')
plt.tight_layout()
plt.show()
print('Failure Rate Matrix (%):')
display(net_dev_failure_rate.round(2))

> **Interpretation (Q14):** Android on 4G carries the highest raw count of failures (5,595) solely because it accounts for the vast majority of all transactions (112,476 tx). The cohort failure rates across all (device, network) combinations remain strictly between 4.79% and 5.34%. Statistical tests confirm no platform is over-represented in failure probability.

**Q15. Which states have the most fraud-flagged transactions?**

In [ ]:
fraud_tx_df = df[df['fraud_flag'] == 1]
fraud_state_count = fraud_tx_df['sender_state'].value_counts()
state_total = df['sender_state'].value_counts()
state_fraud_rate = (fraud_state_count / state_total) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fraud_state_count.plot(kind='bar', ax=ax1, color='#f59e0b', alpha=0.85)
ax1.set_title('Fraud-Flagged Transactions (Count) by State')
ax1.set_ylabel('Number of Flagged Transactions')
ax1.tick_params(axis='x', rotation=30)

state_fraud_rate.reindex(fraud_state_count.index).plot(kind='bar', ax=ax2, color='#be185d', alpha=0.85)
ax2.set_title('Fraud-Flagged Rate (%) by State')
ax2.set_ylabel('Cohort Fraud Rate (%)')
ax2.set_ylim(0, 0.35)
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
corr_cols = ['amount (INR)', 'hour_of_day', 'is_weekend', 'fraud_flag']
corr_matrix = df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-0.05, vmax=0.05)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()
display(corr_matrix)

> **Interpretation:** Pairwise correlations between amount, hour, weekend, and fraud flag are all virtually zero (-0.002 to +0.003). Transaction amount does not linearly predict fraud occurrence in this dataset.

## 4. Machine Learning Experiment: Transaction Failure Prediction

### Pipeline Setup, Feature Encoding & Scaling
We evaluate Logistic Regression for predicting `transaction_status` ('SUCCESS' vs 'FAILED').
**Methodological Improvement:** We apply `StandardScaler` to continuous numerical features (`amount (INR)`) to resolve the optimizer non-convergence warning present in the original notebook.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Select feature set (strictly excluding transaction id)
feature_cols = [
    'transaction type', 'merchant_category', 'amount (INR)',
    'sender_age_group', 'receiver_age_group', 'sender_state',
    'sender_bank', 'receiver_bank', 'device_type',
    'network_type', 'hour_of_day', 'day_of_week', 'is_weekend'
]

X = pd.get_dummies(df[feature_cols], drop_first=True)
y = df['transaction_status']

# Stratified 80-20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize numerical features to guarantee lbfgs convergence
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled['amount (INR)'] = scaler.fit_transform(X_train[['amount (INR)']])
X_test_scaled['amount (INR)'] = scaler.transform(X_test[['amount (INR)']])

print('Feature matrix shape:', X_train_scaled.shape)
print('Target distribution (Test Set):\n', y_test.value_counts())

### Model 1: Baseline Logistic Regression (Unweighted)

In [ ]:
model_base = LogisticRegression(max_iter=1000)
model_base.fit(X_train_scaled, y_train)
y_pred_base = model_base.predict(X_test_scaled)

print('Baseline Accuracy:', accuracy_score(y_test, y_pred_base))
print('\nConfusion Matrix [FAILED, SUCCESS]:')
print(confusion_matrix(y_test, y_pred_base, labels=['FAILED', 'SUCCESS']))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_base, labels=['FAILED', 'SUCCESS'], zero_division=0))

> **Model 1 Evaluation:** The unweighted model achieves **95.05% accuracy**, but its recall for FAILED transactions is **0.00**. Because 95.05% of rows are SUCCESS, the model minimizes loss by predicting SUCCESS for all 50,000 test cases. Accuracy is a completely misleading metric here.

### Model 2: Balanced Logistic Regression (`class_weight='balanced'`)

In [ ]:
model_bal = LogisticRegression(max_iter=1000, class_weight='balanced')
model_bal.fit(X_train_scaled, y_train)
y_pred_bal = model_bal.predict(X_test_scaled)

print('Balanced Model Accuracy:', accuracy_score(y_test, y_pred_bal))
print('\nConfusion Matrix [FAILED, SUCCESS]:')
print(confusion_matrix(y_test, y_pred_bal, labels=['FAILED', 'SUCCESS']))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_bal, labels=['FAILED', 'SUCCESS'], zero_division=0))

### Statistical Interpretation of Model 2 Performance

The balanced model achieved **52.06% accuracy**, **0.49 recall**, and **0.05 precision** for FAILED transactions.

**Critical Scientific Insight:**
- In the preliminary notebook, this was described as the model 'becoming better at detecting failed transactions.'
- In reality, because transaction failure in this dataset is statistically independent of all features (Chi-Square p > 0.27, Cramer's V < 0.004), there is **zero true predictive signal**.
- When class weights are balanced, the model shifts its decision boundary to predict randomly with ~50% probability.
- Random guessing yields **~50% recall** and precision equal to the population base rate (**~5.0%**), with **~50% accuracy**.
- **Conclusion:** Machine learning cannot predict transaction failure from these metadata features because failures occur as an independent random process.

## 5. Summary Conclusion

1. **Payment Velocity:** P2P and P2M drive over 80% of volume; Grocery drives volume while Shopping/Education drive value.
2. **Reliability:** National failure rate is stable at ~4.95% across all 8 banks, 3 devices, and 4 networks.
3. **Security:** Fraud heuristic flags occur at 0.192% baseline; state flag counts scale proportionally with transaction volume.
4. **Modeling:** Logistic regression cannot predict failures because failures are stochastic noise in this synthetic dataset.